# Выполнение ЛР №5: Классификация и регрессия

## Подключение библиотек

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт модулей sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Дополнительные импорты для обработки данных
import warnings
warnings.filterwarnings('ignore')

## Настройка библиотек

In [ ]:
# Настройка стилей визуализации
plt.style.use('default')
sns.set_palette("husl")

# Настройка параметров отображения
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Настройка pandas для отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Настройка numpy для воспроизводимости результатов
np.random.seed(42)

print("Библиотеки успешно импортированы и настроены!")
print(f"Версия pandas: {pd.__version__}")
print(f"Версия numpy: {np.__version__}")
print(f"Версия matplotlib: {plt.matplotlib.__version__}")
print(f"Версия seaborn: {sns.__version__}")

## Задание 1: Классификация kNN на датасете flame

### Формулировка

Выполнить классификацию методом k ближайших соседей на датасете flame:
1. Загрузить и разобрать данные из файла flame.txt
1. Разделить данные flame на обучающий и тестовый наборы
1. Оценить точность для разных значений k от 2 до 20 с использованием кросс-валидации
1. Построить график зависимости точности от k

### Решение

#### 1.1 Загрузка и парсинг данных flame

In [ ]:
# Загрузка данных из файла flame.txt
flame_data_path = '../ЛР (4)/Вариант 4/flame.txt'

# Чтение данных с разделителем табуляция
flame_data = pd.read_csv(flame_data_path, sep='\t', header=None, names=['x1', 'x2', 'class'])

print("Данные flame успешно загружены!")
print(f"Размер датасета: {flame_data.shape}")
print("\nПервые 10 строк:")
print(flame_data.head(10))

print("\nИнформация о данных:")
print(flame_data.info())

print("\nРаспределение классов:")
print(flame_data['class'].value_counts().sort_index())

# Разделение данных на признаки (X) и метки классов (y)
X_flame = flame_data[['x1', 'x2']].values
y_flame = flame_data['class'].values

print(f"\nРазмер матрицы признаков X: {X_flame.shape}")
print(f"Размер вектора меток y: {y_flame.shape}")
print(f"Уникальные классы: {np.unique(y_flame)}")

#### 1.2 Разделение данных на обучающий и тестовый наборы

In [ ]:
# Разделение данных flame на обучающий и тестовый наборы
# Используем фиксированный random_state для воспроизводимости результатов
X_train, X_test, y_train, y_test = train_test_split(
    X_flame, y_flame, 
    test_size=0.3,  # 30% данных для тестирования
    random_state=42,  # Фиксированное значение для воспроизводимости
    stratify=y_flame  # Сохранение пропорций классов
)

print("Данные успешно разделены на обучающий и тестовый наборы!")
print(f"Размер исходного датасета: {X_flame.shape[0]} образцов")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов ({X_train.shape[0]/X_flame.shape[0]*100:.1f}%)")
print(f"Размер тестового набора: {X_test.shape[0]} образцов ({X_test.shape[0]/X_flame.shape[0]*100:.1f}%)")

print("\nРаспределение классов в обучающем наборе:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for class_label, count in zip(unique_train, counts_train):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_train)*100:.1f}%)")

print("\nРаспределение классов в тестовом наборе:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for class_label, count in zip(unique_test, counts_test):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_test)*100:.1f}%)")

# Проверка корректности разделения
total_samples = X_train.shape[0] + X_test.shape[0]
print(f"\nПроверка: {X_train.shape[0]} + {X_test.shape[0]} = {total_samples} (исходно: {X_flame.shape[0]})")
assert total_samples == X_flame.shape[0], "Ошибка: потеря данных при разделении!"

#### 1.3 Оценка точности kNN для разных значений k

In [ ]:
# Оценка точности kNN для разных значений k от 2 до 20
k_range = range(2, 21)  # k от 2 до 20
k_scores = []

print("Оценка точности kNN для разных значений k:")
print("k\tТочность (среднее)\tСтандартное отклонение")
print("-" * 50)

# Цикл оценки для каждого k
for k in k_range:
    # Создание модели kNN
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Кросс-валидация с 5 фолдами
    cv_scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    
    # Сохранение среднего значения точности
    mean_accuracy = cv_scores.mean()
    std_accuracy = cv_scores.std()
    k_scores.append(mean_accuracy)
    
    print(f"{k}\t{mean_accuracy:.4f}\t\t{std_accuracy:.4f}")

print(f"\nВсего оценено k значений: {len(k_scores)}")
print(f"Лучшая точность: {max(k_scores):.4f} при k = {k_range[k_scores.index(max(k_scores))]}")
print(f"Худшая точность: {min(k_scores):.4f} при k = {k_range[k_scores.index(min(k_scores))]}")

#### 1.4 Построение графика точности vs k

In [ ]:
# Построение графика зависимости точности от k
plt.figure(figsize=(12, 8))

# Основной график
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8, label='Точность кросс-валидации')

# Выделение максимального значения
best_k = k_range[k_scores.index(max(k_scores))]
best_score = max(k_scores)
plt.plot(best_k, best_score, 'ro', markersize=12, label=f'Лучший результат (k={best_k})')

# Настройка графика
plt.xlabel('Количество соседей (k)', fontsize=14)
plt.ylabel('Точность классификации', fontsize=14)
plt.title('Зависимость точности kNN от количества соседей k\n(датасет flame)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Установка диапазона осей
plt.xlim(1.5, 20.5)
plt.ylim(min(k_scores) - 0.02, max(k_scores) + 0.02)

# Настройка тиков на оси x
plt.xticks(range(2, 21, 2))

plt.tight_layout()
plt.show()

# Вывод статистики
print(f"\nСтатистика по результатам:")
print(f"Оптимальное значение k: {best_k}")
print(f"Максимальная точность: {best_score:.4f}")
print(f"Средняя точность по всем k: {np.mean(k_scores):.4f}")
print(f"Стандартное отклонение: {np.std(k_scores):.4f}")

#### Выводы по заданию 1

1. **Загрузка данных**: Успешно загружен датасет flame с двумя признаками (x1, x2) и двумя классами (1, 2)
1. **Разделение данных**: Данные flame успешно разделены на обучающий (70%) и тестовый (30%) наборы с сохранением пропорций классов
1. **Оценка kNN**: Проведена оценка точности для k от 2 до 20 с использованием 5-фолдовой кросс-валидации
1. **Визуализация**: Построен график зависимости точности от k, который показывает оптимальное значение k
1. **Результат**: Определено оптимальное значение k для данного датасета

## Задание 2: Визуализация данных с разделением на классы

### Формулировка

Визуализировать обучающие и тестовые данные с метками классов:
1. Создать диаграмму рассеяния с цветовым кодированием по классам
2. Различить обучающие и тестовые данные визуально
3. Добавить легенду с метками классов

### Решение

#### 2.1 Создание диаграммы рассеяния с классами

In [ ]:
# Определение цветов для классов
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# Дополнительная детальная визуализация
plt.figure(figsize=(12, 8))

# Создание более детального графика с четким разделением
for class_label in np.unique(y_flame):
    # Обучающие данные для каждого класса
    mask_train = y_train == class_label
    plt.scatter(X_train[mask_train, 0], X_train[mask_train, 1], 
               c=colors[class_label], alpha=0.7, s=70, 
               label=f'{class_names[class_label]} - Обучение ({np.sum(mask_train)} точек)', 
               marker='o', edgecolors='darkgray', linewidth=0.8)
    
    # Тестовые данные для каждого класса
    mask_test = y_test == class_label
    plt.scatter(X_test[mask_test, 0], X_test[mask_test, 1], 
               c=colors[class_label], alpha=1.0, s=100, 
               label=f'{class_names[class_label]} - Тест ({np.sum(mask_test)} точек)', 
               marker='^', edgecolors='black', linewidth=1.2)

plt.xlabel('Признак x1', fontsize=14)
plt.ylabel('Признак x2', fontsize=14)
plt.title('Детальная визуализация данных flame с разделением на классы и наборы', fontsize=16)
plt.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

#### Выводы по заданию 2

1. **Визуализация классов**: Создана диаграмма рассеяния с цветовым кодированием по классам (красный - класс 1, синий - класс 2)
1. **Различение наборов**: Обучающие данные показаны кругами, тестовые - треугольниками для четкого визуального различения

## Задание 3: Анализ результатов классификации

### Формулировка

Проанализировать результаты классификации kNN:
1. Выбрать оптимальное значение k на основе результатов кросс-валидации
1. Обучить финальную модель kNN с оптимальным k
1. Построить матрицу ошибок (confusion matrix)

### Решение

#### 3.1 Обучение оптимальной модели kNN

In [ ]:
# Выбор оптимального значения k на основе результатов кросс-валидации
optimal_k = k_range[k_scores.index(max(k_scores))]
optimal_accuracy = max(k_scores)

print("=== ВЫБОР ОПТИМАЛЬНОГО ЗНАЧЕНИЯ K ===")
print(f"Оптимальное значение k: {optimal_k}")
print(f"Максимальная точность кросс-валидации: {optimal_accuracy:.4f}")

# Обучение финальной модели kNN с оптимальным k
print("\n=== ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ ===")
final_knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn_model.fit(X_train, y_train)

print(f"Модель kNN успешно обучена с k = {optimal_k}")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Количество классов: {len(np.unique(y_train))}")

# Проверка обученной модели
print("\n=== ПАРАМЕТРЫ ОБУЧЕННОЙ МОДЕЛИ ===")
print(f"Алгоритм: {final_knn_model.algorithm}")
print(f"Метрика расстояния: {final_knn_model.metric}")
print(f"Количество соседей: {final_knn_model.n_neighbors}")
print(f"Веса: {final_knn_model.weights}")

# Получение предсказаний на тестовом наборе
y_pred = final_knn_model.predict(X_test)
y_pred_proba = final_knn_model.predict_proba(X_test)

print("\n=== ПРЕДСКАЗАНИЯ НА ТЕСТОВОМ НАБОРЕ ===")
print(f"Количество тестовых образцов: {len(y_test)}")
print(f"Количество предсказаний: {len(y_pred)}")
print(f"Уникальные предсказанные классы: {np.unique(y_pred)}")
print(f"Уникальные истинные классы: {np.unique(y_test)}")

# Предварительная оценка точности
test_accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочность на тестовом наборе: {test_accuracy:.4f}")
print(f"Разница с кросс-валидацией: {abs(test_accuracy - optimal_accuracy):.4f}")

#### 3.2 Вычисление метрик производительности

In [ ]:
# Построение матрицы ошибок (confusion matrix)
print("\n=== МАТРИЦА ОШИБОК ===")
cm = confusion_matrix(y_test, y_pred)
print("Матрица ошибок (строки - истинные классы, столбцы - предсказанные):")
print(cm)

# Визуализация матрицы ошибок
plt.figure(figsize=(10, 8))

# Создание heatmap для матрицы ошибок
unique_classes = np.unique(y_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Предсказан\nКласс {i}' for i in unique_classes],
            yticklabels=[f'Истинный\nКласс {i}' for i in unique_classes],
            cbar_kws={'label': 'Количество образцов'})

plt.title(f'Матрица ошибок для kNN (k={optimal_k})', fontsize=16)
plt.xlabel('Предсказанный класс', fontsize=14)
plt.ylabel('Истинный класс', fontsize=14)

# Добавление процентов в ячейки
total_samples = np.sum(cm)
for i in range(len(unique_classes)):
    for j in range(len(unique_classes)):
        percentage = cm[i, j] / total_samples * 100
        plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

# Анализ матрицы ошибок
print("\n--- АНАЛИЗ МАТРИЦЫ ОШИБОК ---")
total_correct = np.trace(cm)  # Сумма диагональных элементов
total_samples = np.sum(cm)
total_errors = total_samples - total_correct

print(f"Всего образцов: {total_samples}")
print(f"Правильно классифицировано: {total_correct} ({total_correct/total_samples*100:.1f}%)")
print(f"Неправильно классифицировано: {total_errors} ({total_errors/total_samples*100:.1f}%)")

#### Выводы по заданию 3

1. **Оптимальная модель**: Выбрано оптимальное значение k на основе кросс-валидации и обучена финальная модель kNN
3. **Матрица ошибок**: Построена и проанализирована confusion matrix

## Задание 4

### Формулировка

Проанализировать и визуализировать ошибки классификации kNN:
1. Идентифицировать неправильно классифицированные точки
1. Выделить эти точки на графике
1. Показать границы принятия решений

### Решение

#### 4.1 Идентификация неправильно классифицированных точек

In [ ]:
# Идентификация неправильно классифицированных точек
print("=== ИДЕНТИФИКАЦИЯ ОШИБОК КЛАССИФИКАЦИИ ===")

# Найти индексы неправильно классифицированных образцов
misclassified_mask = y_test != y_pred
misclassified_indices = np.where(misclassified_mask)[0]
correctly_classified_indices = np.where(~misclassified_mask)[0]

print(f"Всего тестовых образцов: {len(y_test)}")
print(f"Правильно классифицировано: {len(correctly_classified_indices)} ({len(correctly_classified_indices)/len(y_test)*100:.1f}%)")
print(f"Неправильно классифицировано: {len(misclassified_indices)} ({len(misclassified_indices)/len(y_test)*100:.1f}%)")

# Сохранение данных об ошибках для дальнейшего использования
misclassified_points = X_test[misclassified_indices]
misclassified_true_labels = y_test[misclassified_indices]
misclassified_pred_labels = y_pred[misclassified_indices]

correctly_classified_points = X_test[correctly_classified_indices]
correctly_classified_labels = y_test[correctly_classified_indices]

print(f"\nДанные об ошибках сохранены для визуализации:")
print(f"- Неправильно классифицированные точки: {misclassified_points.shape}")
print(f"- Правильно классифицированные точки: {correctly_classified_points.shape}")

#### 4.2 Создание визуализации ошибок

In [ ]:
# Создание визуализации ошибок классификации
print("=== ВИЗУАЛИЗАЦИЯ ОШИБОК КЛАССИФИКАЦИИ ===")

# Создание фигуры
fig, axes = plt.subplots(1, 1, figsize=(10, 8))

# Определение цветов для классов и типов предсказаний
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# График 1: Границы принятия решений
ax1 = axes

# Создание сетки для визуализации границ решений
h = 0.02  # Шаг сетки
x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Предсказания для всех точек сетки
mesh_points = np.c_[xx.ravel(), yy.ravel()]
Z = final_knn_model.predict(mesh_points)
Z = Z.reshape(xx.shape)

# Отображение границ решений
ax1.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
ax1.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)

# Отображение тестовых точек
for class_label in np.unique(y_test):
    mask = y_test == class_label
    ax1.scatter(X_test[mask, 0], X_test[mask, 1], 
               c=colors[class_label], alpha=0.8, s=60, 
               label=f'{class_names[class_label]}', 
               marker='o', edgecolors='black', linewidth=0.5)

# Выделение ошибок
if len(misclassified_indices) > 0:
    ax1.scatter(misclassified_points[:, 0], misclassified_points[:, 1], 
               c='yellow', alpha=1.0, s=150, 
               label=f'Ошибки ({len(misclassified_indices)})', 
               marker='X', edgecolors='black', linewidth=2)

ax1.set_xlabel('Признак x1')
ax1.set_ylabel('Признак x2')
ax1.set_title('Границы принятия решений kNN')
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Выводы по заданию 4

1. **Идентификация ошибок**: Найдены и проанализированы все неправильно классифицированные точки с детальной статистикой
1. **Визуализация ошибок**: Создана визуализация границы принятия решений kNN с выделением ошибок

## Задание 5

### Формулировка

### Решение

## Задание 6

### Формулировка

### Решение

## Задание 7

### Формулировка

### Решение

## Задание 8

### Формулировка

### Решение